# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I am building a **Randon Forest Classifier** as the primary model while using **Logistic Regression** and **Descision Tree** as simple comaprisons.

* Why this fits: Random forest is excellent for non linear patterns like search and analystics data it is also good at handling interactions between signals like in my data combining high impressions with low clich rates.

* Why classification for ranking: though the final goal is to rank pages, i will train a classifier to predict probability of decline a score of 0.0 and 1.0 then sort the pages by probability.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I will hold out 20% of the clients during spliting using the **Client-Holdout Split ** rather than a simple random row split

* Why the split is honest: in the dataset, each page belongs to a specific client (website). Pages on the same website share similar hidden patterns like domain authourity, industry niche and website design. If we split rows randomly, the model will memorise specific patterns and score artificially high. Client-Holdout split test the model on data it has never seen before.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [1]:
import pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score

In [2]:
# load data

data = pd.read_csv("../../data/processed/refresh_feature_vector.csv")
baseline_data = pd.read_csv("../../data/processed/baseline_refresh_queue.csv")

In [3]:
data.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,trend_direction,trend_pct,is_declining_label,log_impressions_90d,log_clicks_90d,log_sessions_90d,log_ai_sessions_90d,has_clicks,has_ai_sessions,measurable_opportunity
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,down,-41.4,1,8.243808,3.401197,2.890372,0.0,1,0,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,down,-57.7,1,9.636980,2.079442,2.302585,0.0,1,0,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,down,-60.9,1,9.440023,2.484907,2.484907,0.0,1,0,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,0.0,0.0,...,stable,-13.8,0,9.371779,4.077537,4.369448,0.0,1,0,1
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,down,-34.7,1,9.859588,3.218876,4.983607,0.0,1,0,1


In [4]:
baseline_data.head()

,content_id,client_id,baseline_rank,baseline_refresh_score,visibility_score,freshness_risk_score,position_opportunity_score,depth_gap_score,reason_codes,suggested_action_baseline,...,clicks_90d,sessions_90d,avg_position,ctr,engagement_rate,scroll_rate,content_age_days,days_since_last_update,word_count,trend_direction
0,content_9532f197bbc8,client_4e07408562,1,0.941189,0.999633,0.8432,0.979233,0.871347,declining_with_demand|page_one_decay_risk|low_...,refresh,...,2689,1098,2.0,0.87,8.01,28.75,445,104,0.0,down
1,content_4d1fe5b32dc2,client_19581e27de,2,0.934889,0.994167,0.8432,0.963733,0.866582,page_one_decay_risk|low_engagement_visible_page,monitor,...,512,549,2.5,0.52,7.47,13.15,329,104,0.0,stable
2,content_07f2e7a6f38a,client_19581e27de,3,0.934080,0.994467,0.8432,0.959965,0.866843,page_one_decay_risk|low_engagement_visible_page,monitor,...,856,780,2.7,0.85,2.05,4.60,313,104,0.0,stable
3,content_e5ae436f9a16,client_4e07408562,4,0.933606,0.996000,0.8432,0.955347,0.868180,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,...,533,522,3.0,0.45,7.09,12.60,421,104,0.0,stable
4,content_3430a8b94511,client_19581e27de,5,0.933559,0.998167,0.8432,0.951314,0.870069,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,...,440,534,3.3,0.29,6.18,11.04,329,104,0.0,stable


In [5]:
numeric_features = [ "search_volume", "competition", "cpc", "word_count", "char_count", "log_impressions_90d",
"log_clicks_90d", "log_sessions_90d", "days_with_impressions", "days_with_sessions", 
"content_age_days", "days_since_last_update", "ctr", "avg_position", "engagement_rate",
"scroll_rate", "ai_traffic_pct"]

categorical_features = [ "competition_level", "content_type", "main_intent", "age_tier", "freshness_tier", 
"word_count_tier", "impression_tier", "position_tier"]


X_num = data[numeric_features].fillna(0)
X_cat = pd.get_dummies(data[categorical_features].fillna("unknow"), dtype=float)

X = pd.concat([X_num, X_cat], axis=1)
y = data['is_declining_label'].astype(int)

In [6]:
# Perform the client holder split 
unique_client = data['client_id'].unique()
np.random.seed(42)

shuffle_client = np.random.permutation(unique_client)
test_client_count = int(round(len(shuffle_client) * 0.2 ))
test_client = set(shuffle_client[:test_client_count])

test_mask = data['client_id'].isin(test_client).to_numpy()

train_idx = np.where(~test_mask)[0]
test_idx = np.where(test_mask)[0]

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]

y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

In [8]:
def prescision_at_50(y_true, predicted):
    frame = pd.DataFrame({ "y_true" : list(y_true), "predicted" : list(predicted)})
    top = frame.sort_values("predicted", ascending=False).head(50)

    return top["y_true"].mean()

In [9]:
# logistic regression model
scaler = StandardScaler()
X_trained_scaler = scaler.fit_transform(X_train)
X_test_scaler = scaler.transform(X_test)

lr_model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
lr_model.fit(X_trained_scaler, y_train)
y_proba_lg = lr_model.predict_proba(X_test_scaler)[:, 1]

lr_roc_auc = roc_auc_score(y_test, y_proba_lg)
lr_prescision = prescision_at_50(y_test, y_proba_lg)

print(f"Logistic Regression ROC-AUC: {lr_roc_auc}")
print(f"Logistic Regression Prescision: {lr_prescision}")

Logistic Regression ROC-AUC: 0.6517653868415975
Logistic Regression Prescision: 0.76


In [10]:
# Decision tree model 

dt_model = DecisionTreeClassifier(
    class_weight="balanced", max_depth=5, min_samples_leaf=50, random_state=42
)

dt_model.fit(X_train, y_train)

y_proba_dt = dt_model.predict_proba(X_test)[:, 1]

dt_roc_auc = roc_auc_score(y_test, y_proba_dt)
dt_precision = prescision_at_50(y_test, y_proba_dt)

print(f'Descision Tree ROC-AUC: {dt_roc_auc}')
print(f"Descision Tree Precision: {dt_precision}")

Descision Tree ROC-AUC: 0.6639871608229702
Descision Tree Precision: 0.72


In [11]:
# Random Forest model
rf_model = RandomForestClassifier(
    class_weight="balanced_subsample", max_depth=15, min_samples_leaf=25, 
    n_estimators= 200, n_jobs=1, random_state=42
)

rf_model.fit(X_train, y_train)

y_proba_rf = rf_model.predict_proba(X_test)[:, 1]
rf_roc_auc = roc_auc_score(y_test, y_proba_rf)
rf_precision = prescision_at_50(y_test, y_proba_rf)

print(f"Random Forest ROC-AUC: {rf_roc_auc}")
print(f"Random Forest Precision: {rf_precision}")

Random Forest ROC-AUC: 0.6689565537684388
Random Forest Precision: 0.58


In [12]:
# the baseline model

baseline_look = baseline_data.set_index("content_id")['baseline_refresh_score']
baseline_score_test = data.iloc[test_idx]['content_id'].map(baseline_look).fillna(0).to_numpy()

baseline_roc_auc = roc_auc_score(y_test, baseline_score_test)
baseline_precision = prescision_at_50(y_test, baseline_score_test)

print(f"Baseline ROC-AUC: {baseline_roc_auc}")
print(f"Baseline Precision: {baseline_precision}")

Baseline ROC-AUC: 0.587634048374932
Baseline Precision: 0.48


In [13]:
comparision_data = {
    "Model Name" : ['BaseLine Model', 'Logistic Regression Model', "Decision Tree Model", "Random Forest Model"],
    "ROC-AUC" : [baseline_roc_auc, lr_roc_auc, dt_roc_auc, rf_roc_auc],
    "Precision@50" : [baseline_precision, lr_prescision, dt_precision, rf_precision ],
}

comparision = pd.DataFrame(comparision_data)
comparision

,Model Name,ROC-AUC,Precision@50
0,BaseLine Model,0.587634,0.48
1,Logistic Regression Model,0.651765,0.76
2,Decision Tree Model,0.663987,0.72
3,Random Forest Model,0.668957,0.58


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### My Finding on Model Performance: 
Surprisingly, Logistic Regression achieved the highest Precision@50 (76%) on the client-holdout test set, outperforming the Random Forest (58%) and Decision Tree (72%).

Why this happened:

Client-Holdout Generalization: Because our test split evaluates the models on websites (clients) they have never seen before, simplicity is a massive advantage. Logistic Regression is less complex and has lower variance, making it highly robust when generalizing to new domain patterns.
Random Forest Overfitting: Even with a reduced depth (max_depth=10), the Random Forest is prone to memorizing combinations of features (like content_type and search_volume) that are unique to the training clients but do not translate to the test clients.

I will now continue moving forward with Logistic Regression as my model. It is simpler faster and more interpretable for editors.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.